In [3]:
# --------------------------------------------
#  Trompo de Lagrange – versión "Pphi/Ppsi"
#  (c) 2025
# --------------------------------------------
import numpy as np
from numba import njit
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, HTMLWriter
from mpl_toolkits.mplot3d import Axes3D      # noqa: F401

# -----------------------------
#  1. Parámetros físicos
# -----------------------------
m = 0.1          # masa [kg]
r = 0.1          # radio del disco [m]
d = 0.3         # CM-punta [m]
g = 9.81        # gravedad [m/s²]

# -----------------------------
#  2. Condiciones iniciales
# -----------------------------
theta0    = np.deg2rad(30)   # inclinación inicial
w0        = 350.0            # giro inicial alrededor del eje (rad/s)
phi0      = 0.0              # fase de precesión
psi0      = 0.0              # fase de giro propio
dtheta0   = 0.0              # velocidad polar inicial

# Estado inicial y = [phi, psi, theta, dtheta]
y0 = np.array([phi0, psi0, theta0, dtheta0], dtype=np.float64)

# -----------------------------
#  3. Ecuaciones de movimiento
# -----------------------------
# Pre-cálculo de momentos de inercia
I0 = 0.25*m*r*r + m*d*d      # respecto de un eje perpendicular al disco por el CM
Iz = 0.5*m*r*r               # respecto del eje de simetría

# Constantes de movimiento que vienen de tu formulación
Pphi = Iz*w0*np.cos(theta0)
Ppsi = Iz*w0

@njit(fastmath=True)
def rhs(t, y):
    """Ecuaciones diferenciales tal como las proporcionaste."""
    phi, psi, theta, dtheta = y
    s, c = np.sin(theta), np.cos(theta)

    Ip = (I0*s*s + Iz*c*c)*Iz - Iz*Iz*c*c
    phi_dot = (Iz*Pphi - Iz*c*Ppsi) / Ip
    psi_dot = ((I0*s*s + Iz*c*c)*Ppsi - Iz*c*Pphi) / Ip

    dtheta_dot = ((phi_dot**2)*s*c*(I0 - Iz)
                  - phi_dot*psi_dot*Iz*s
                  + m*g*d*s) / I0

    return np.array([phi_dot,
                     psi_dot,
                     dtheta,        # dθ/dt
                     dtheta_dot], dtype=np.float64)

# -----------------------------
#  4. Integración numérica
# -----------------------------
t_max  = 6
t_eval = np.linspace(0.0, t_max, 200)   # 1 ms de paso de guardado

sol = solve_ivp(rhs, (0.0, t_max), y0,
                method="DOP853", t_eval=t_eval,
                rtol=1e-9, atol=1e-12, max_step=1e-3)

phi, psi, theta = sol.y[0], sol.y[1], sol.y[2]
t               = sol.t

# -----------------------------
#  5. Animación 3D
# -----------------------------
fig = plt.figure(figsize=(8, 4))
ax3d = fig.add_subplot(121, projection='3d', facecolor='whitesmoke')
axT  = fig.add_subplot(122,  facecolor='whitesmoke')

# Configuración ejes
lim = d*1.1
for ax in (ax3d,):
    ax.set_xlim([-lim, lim]); ax.set_ylim([-lim, lim]); ax.set_zlim([0, 2*lim])
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.view_init(elev=25, azim=45)

axT.set_xlim([0, t_max]); axT.set_ylim([0, np.pi])
axT.set_xlabel('t [s]'); axT.set_ylabel(r'$\theta$ [rad]')

# Elementos gráficos
line3d,  = ax3d.plot([], [], [], 'r-', lw=2)
tip3d,   = ax3d.plot([], [], [], 'ko', ms=6)
trace3d, = ax3d.plot([], [], [], 'b--', lw=1, alpha=0.6)
lineT,   = axT.plot([], [], 'r-')

trace_x, trace_y, trace_z = [], [], []

def init():
    line3d.set_data_3d([], [], [])
    tip3d.set_data_3d([], [], [])
    trace3d.set_data_3d([], [], [])
    lineT.set_data([], [])
    return line3d, tip3d, trace3d, lineT

def update(i):
    x = d*np.sin(theta[i])*np.cos(phi[i])
    y = d*np.sin(theta[i])*np.sin(phi[i])
    z = d*np.cos(theta[i])

    line3d.set_data_3d([0, x], [0, y], [0, z])
    tip3d.set_data_3d([x], [y], [z])

    trace_x.append(x); trace_y.append(y); trace_z.append(z)
    trace3d.set_data_3d(trace_x, trace_y, trace_z)

    lineT.set_data(t[:i+1], theta[:i+1])
    return line3d, tip3d, trace3d, lineT

ani = FuncAnimation(fig, update, frames=range(0, len(t), 1),
                    init_func=init, interval=20, blit=True)
plt.close()
# Para Jupyter / Colab
HTMLWriter.embed_limit = 40  # MB
from IPython.display import HTML
HTML(ani.to_html5_video())
